In [ ]:
%pip install -q numpy matplotlib transformers sagemaker boto3 pandas
# torch is pre-installed in SageMaker PyTorch kernels; install separately if using a different kernel:
# %pip install -q torch

# Week 2 Wednesday -- Applied ML Transformers + SageMaker Experiments and Lineage Tracking

Friday's encoder-decoder compressed the entire input into a single context vector. Monday's LSTMs solved vanishing gradients but still process tokens one at a time -- a sequential bottleneck that wastes GPU parallelism. Today you learn **transformers**, the architecture that solves both problems with **attention**: parallel processing and direct connections to every input position.

By the end of this notebook you will have:

1. **Built scaled dot-product attention from scratch** in NumPy -- queries, keys, values, and the attention weight heatmap.
2. **Extended to multi-head attention** and **positional encoding** to assemble a complete transformer encoder block.
3. **Used pre-trained transformers** (BERT, GPT, T5) via Hugging Face for classification, generation, and summarization -- the text equivalent of Friday's MobileNet fine-tuning.
4. **Tracked experiments** with SageMaker Experiments, connected lineage to Tuesday's Feature Store, and established reproducibility patterns.

| Block | Content | Minutes |
|-------|---------|--------|
| Stage 1 | Attention and Transformer Architecture | 55 |
| Break 1 | Stretch / Questions | 5 |
| Stage 2 | Pre-trained Models and Fine-tuning | 55 |
| Break 2 | Stretch / Questions | 5 |
| Stage 3 | Experiments, Lineage, and Reproducibility | 55 |

## Setup

Run the cell below to establish your SageMaker session. This connects to your execution role, default S3 bucket, and region. Stage 3 (Experiments and Lineage) requires these AWS credentials.

In [ ]:
import boto3
import sagemaker

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name
bucket = session.default_bucket()
prefix = "fraudshield"

print(f"Region:  {region}")
print(f"Bucket:  s3://{bucket}")
print(f"Role:    ...{role[-30:]}")

---
# STAGE 1 -- Attention and Transformer Architecture

**Connecting to prior days:**

- **Friday:** You built an encoder-decoder that compressed an entire input sequence into a single context vector. That vector is a bottleneck -- long inputs lose information.
- **Monday:** You opened the LSTM black box and saw how gates solve vanishing gradients. But LSTMs still process tokens sequentially: step 50 cannot start until steps 1-49 finish.

**What transformers fix:**

| Problem | LSTM Solution | Transformer Solution |
|---------|--------------|---------------------|
| Vanishing gradients | Cell state highway (addition) | Residual connections + short attention paths |
| Sequential bottleneck | None -- inherently sequential | Self-attention computes all positions in parallel |
| Information bottleneck | None -- single context vector | Every position attends to every other position |

We build the transformer encoder block from scratch, one component at a time.

## Motivating Attention: The Encoder-Decoder Bottleneck

Recall Friday's encoder-decoder architecture:

```
Without attention:
  Encoder: x_1, x_2, ..., x_n --> [LSTM] --> single context vector --> Decoder
```

The entire input sequence is squeezed into one fixed-size vector. For a 200-word paragraph, that vector must encode everything -- and inevitably drops detail.

**Attention** solves this by letting the decoder look at ALL encoder hidden states at each generation step:

```
With attention:
  Encoder: x_1, x_2, ..., x_n --> [LSTM] --> h_1, h_2, ..., h_n
                                                    |  |       |
  Decoder at each step: ---------> weighted sum of ALL encoder states
```

**Self-attention** (used in transformers) goes further: it removes the LSTM entirely. Every token computes relationships with every other token in the same sequence, in parallel.

> **Discussion:** Why is the sequential processing of LSTMs a problem for modern hardware? (GPUs have thousands of cores designed for parallel matrix operations. Processing one token at a time uses a single core while thousands sit idle.)

## Scaled Dot-Product Attention

The attention mechanism has three inputs:

- **Query (Q):** "What am I looking for?" -- the current position's question
- **Key (K):** "What do I contain?" -- each position's identifier
- **Value (V):** "What information do I carry?" -- each position's content

The formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Step by step:
1. Compute similarity scores: $QK^T$ (dot product between each query and all keys)
2. Scale by $\sqrt{d_k}$ to prevent softmax saturation
3. Apply softmax to get attention weights (probability distribution)
4. Multiply weights by values to get the output

> **What is happening in the next cell:** We implement `softmax` and `scaled_dot_product_attention` from scratch in NumPy, then run a 4-token demo with `d_k = 8`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def softmax(x):
    """Numerically stable softmax along the last axis."""
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)


def scaled_dot_product_attention(Q, K, V):
    """Compute scaled dot-product attention.
    
    Args:
        Q: queries, shape (n_queries, d_k)
        K: keys,    shape (n_keys,    d_k)
        V: values,  shape (n_keys,    d_v)
    
    Returns:
        output:  weighted sum of values, shape (n_queries, d_v)
        weights: attention weights,      shape (n_queries, n_keys)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T
    scaled_scores = scores / np.sqrt(d_k)
    weights = softmax(scaled_scores)
    output = weights @ V
    return output, weights


np.random.seed(42)
n_tokens = 4
d_k = 8
d_v = 8

Q = np.random.randn(n_tokens, d_k)
K = np.random.randn(n_tokens, d_k)
V = np.random.randn(n_tokens, d_v)

token_labels = ["The", "fraud", "was", "detected"]

output, weights = scaled_dot_product_attention(Q, K, V)

print("Attention Weights (each row sums to 1.0):")
print(f"{'':>10}", end="")
for label in token_labels:
    print(f"{label:>10}", end="")
print()
for i, label in enumerate(token_labels):
    print(f"{label:>10}", end="")
    for j in range(n_tokens):
        print(f"{weights[i, j]:>10.4f}", end="")
    print(f"  (sum = {weights[i].sum():.4f})")

print(f"\nOutput shape: {output.shape}")
print(f"Each output row is a weighted combination of the Value vectors.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(weights, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(n_tokens))
ax.set_yticks(range(n_tokens))
ax.set_xticklabels(token_labels)
ax.set_yticklabels(token_labels)
ax.set_xlabel("Key (attended to)")
ax.set_ylabel("Query (attending from)")
ax.set_title("Scaled Dot-Product Attention Weights")

for i in range(n_tokens):
    for j in range(n_tokens):
        ax.text(j, i, f"{weights[i, j]:.3f}", ha="center", va="center",
                color="white" if weights[i, j] > 0.5 else "black", fontsize=10)

plt.colorbar(im, ax=ax, label="Attention Weight")
plt.tight_layout()
plt.show()

## Why Scaling Matters

The division by $\sqrt{d_k}$ is not optional decoration. When $d_k$ is large, the dot products $QK^T$ have high variance, pushing softmax into regions where the output is nearly one-hot and gradients are near zero.

> **What is happening in the next cell:** We compare the raw attention scores and softmax outputs for $d_k = 8$ (small) versus $d_k = 512$ (large), both with and without scaling. Notice how unscaled scores with large $d_k$ produce a collapsed distribution.

In [ ]:
np.random.seed(7)

n = 4

Q_small = np.random.randn(n, 8)
K_small = np.random.randn(n, 8)
Q_large = np.random.randn(n, 512)
K_large = np.random.randn(n, 512)

scores_small = Q_small @ K_small.T
scores_large = Q_large @ K_large.T

print("=" * 60)
print("RAW SCORE STATISTICS (before scaling)")
print("=" * 60)
print(f"d_k =   8:  mean = {scores_small.mean():>7.3f},  std = {scores_small.std():>7.3f}")
print(f"d_k = 512:  mean = {scores_large.mean():>7.3f},  std = {scores_large.std():>7.3f}")
print()
print("SOFTMAX WITHOUT SCALING:")
print(f"  d_k =   8:  {softmax(scores_small)[0].round(4)}")
print(f"  d_k = 512:  {softmax(scores_large)[0].round(4)}  <-- near one-hot!")
print()
print("SOFTMAX WITH SCALING (divided by sqrt(d_k)):")
scaled_small = scores_small / np.sqrt(8)
scaled_large = scores_large / np.sqrt(512)
print(f"  d_k =   8:  {softmax(scaled_small)[0].round(4)}")
print(f"  d_k = 512:  {softmax(scaled_large)[0].round(4)}  <-- reasonable distribution")
print()
print("Scaling keeps the variance of the scores near 1 regardless of d_k,")
print("preventing softmax saturation and maintaining healthy gradients.")

## Multi-Head Attention

A single attention head computes one set of weights -- one way of relating tokens. But tokens need to relate in multiple ways simultaneously:

- **Head 1** might learn syntactic dependencies (subject-verb agreement)
- **Head 2** might learn semantic relationships ("fraud" relates to "detected")
- **Head 3** might learn positional patterns (adjacent tokens)

Multi-head attention runs several attention computations in parallel, each in a lower-dimensional subspace:

| Step | Operation | Shapes (d_model=32, n_heads=4) |
|------|-----------|-------------------------------|
| 1 | Project Q, K, V | (n_tokens, 32) each |
| 2 | Split into heads | 4 sets of (n_tokens, 8) each |
| 3 | Attention per head | 4 outputs of (n_tokens, 8) |
| 4 | Concatenate heads | (n_tokens, 32) |
| 5 | Output projection (W_O) | (n_tokens, 32) |

The total parameter count is the same as single-head attention -- we just partition the dimensions.

> **What is happening in the next cell:** We implement multi-head attention in NumPy. The function splits Q, K, V into `n_heads` pieces, computes attention independently per head, concatenates the results, and applies an output projection.

In [ ]:
def multi_head_attention(Q, K, V, n_heads, W_O):
    """Multi-head attention.
    
    Args:
        Q, K, V: shape (n_tokens, d_model)
        n_heads: number of attention heads
        W_O:     output projection, shape (d_model, d_model)
    
    Returns:
        output:      shape (n_tokens, d_model)
        all_weights: list of per-head attention weight matrices
    """
    n_tokens, d_model = Q.shape
    d_head = d_model // n_heads

    Q_heads = Q.reshape(n_tokens, n_heads, d_head).transpose(1, 0, 2)
    K_heads = K.reshape(n_tokens, n_heads, d_head).transpose(1, 0, 2)
    V_heads = V.reshape(n_tokens, n_heads, d_head).transpose(1, 0, 2)

    head_outputs = []
    all_weights = []
    for h in range(n_heads):
        out_h, w_h = scaled_dot_product_attention(Q_heads[h], K_heads[h], V_heads[h])
        head_outputs.append(out_h)
        all_weights.append(w_h)

    concat = np.concatenate(head_outputs, axis=-1)  # (n_tokens, d_model)
    output = concat @ W_O
    return output, all_weights


np.random.seed(42)
d_model = 32
n_heads = 4
n_tokens = 4

Q_mh = np.random.randn(n_tokens, d_model)
K_mh = np.random.randn(n_tokens, d_model)
V_mh = np.random.randn(n_tokens, d_model)
W_O = np.random.randn(d_model, d_model) * 0.01

mh_output, head_weights = multi_head_attention(Q_mh, K_mh, V_mh, n_heads, W_O)

print(f"Input shape:  ({n_tokens}, {d_model})")
print(f"Output shape: {mh_output.shape}")
print(f"Heads:        {n_heads}")
print(f"d_head:       {d_model // n_heads}")
print()

fig, axes = plt.subplots(1, n_heads, figsize=(16, 4))
for h in range(n_heads):
    im = axes[h].imshow(head_weights[h], cmap="Blues", vmin=0, vmax=1)
    axes[h].set_title(f"Head {h + 1}")
    axes[h].set_xticks(range(n_tokens))
    axes[h].set_yticks(range(n_tokens))
    axes[h].set_xticklabels(token_labels, fontsize=8)
    axes[h].set_yticklabels(token_labels, fontsize=8)
    for i in range(n_tokens):
        for j in range(n_tokens):
            axes[h].text(j, i, f"{head_weights[h][i, j]:.2f}", ha="center", va="center",
                         color="white" if head_weights[h][i, j] > 0.5 else "black", fontsize=8)

fig.suptitle("Multi-Head Attention Weights (each head learns different patterns)", fontsize=12)
plt.tight_layout()
plt.show()

print("Each head attends to different aspects of the input.")
print("Notice the weight patterns differ across heads -- this is the power of multi-head attention.")

## Positional Encoding

Self-attention treats all positions equally -- it has no built-in notion of token order. Without positional information, "dog bites man" and "man bites dog" produce identical attention patterns.

The original transformer uses **sinusoidal positional encoding**: each position gets a unique vector computed from sine and cosine functions at different frequencies.

$$PE(\text{pos}, 2i) = \sin\left(\frac{\text{pos}}{10000^{2i/d_{\text{model}}}}\right)$$

$$PE(\text{pos}, 2i+1) = \cos\left(\frac{\text{pos}}{10000^{2i/d_{\text{model}}}}\right)$$

- Low-frequency dimensions (large $i$) encode coarse position (beginning vs end of sequence)
- High-frequency dimensions (small $i$) encode fine position (adjacent tokens)
- Every position gets a unique signature
- Relative positions are captured through linear relationships between encodings

> **What is happening in the next cell:** We implement sinusoidal positional encoding and visualize it as a heatmap. Each row is a position, each column is a dimension. The wave patterns show how different dimensions oscillate at different frequencies.

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    """Generate sinusoidal positional encoding matrix.
    
    Args:
        max_len: maximum sequence length
        d_model: model dimension
    
    Returns:
        PE matrix of shape (max_len, d_model)
    """
    PE = np.zeros((max_len, d_model))
    positions = np.arange(max_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))

    PE[:, 0::2] = np.sin(positions * div_term)
    PE[:, 1::2] = np.cos(positions * div_term)
    return PE


max_len = 64
d_model_pe = 32
PE = sinusoidal_positional_encoding(max_len, d_model_pe)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(PE, cmap="RdBu", aspect="auto", vmin=-1, vmax=1)
axes[0].set_xlabel("Dimension")
axes[0].set_ylabel("Position")
axes[0].set_title("Sinusoidal Positional Encoding")
plt.colorbar(im0, ax=axes[0])

for dim_idx in [0, 1, 4, 8, 16]:
    axes[1].plot(PE[:, dim_idx], label=f"dim {dim_idx}")
axes[1].set_xlabel("Position")
axes[1].set_ylabel("Encoding Value")
axes[1].set_title("Selected Dimensions Across Positions")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Positional encoding shape: {PE.shape}")
print(f"Each position has a unique {d_model_pe}-dimensional vector.")
print(f"Low dimensions (0, 1) oscillate rapidly -- fine position.")
print(f"High dimensions (16+) oscillate slowly -- coarse position.")

## Full Transformer Encoder Block -- Summary

We now have all the components. A single transformer encoder block assembles them:

```
Input Embeddings + Positional Encoding
            |
    [Multi-Head Self-Attention]
            |
    + Residual Connection (add input back)
            |
    [Layer Normalization]
            |
    [Feed-Forward Network: Linear -> ReLU -> Linear]
            |
    + Residual Connection (add pre-FFN input back)
            |
    [Layer Normalization]
            |
        Output
```

**Key design choices:**

| Component | Purpose | Analogy to prior days |
|-----------|---------|----------------------|
| Multi-head self-attention | Relate all positions to each other | Replaces LSTM recurrence (Monday) |
| Residual connections | Let gradients flow through deep stacks | Same idea as LSTM cell state highway (Monday) |
| Layer normalization | Stabilize training across dimensions | Standard deep learning technique |
| Feed-forward network | Add per-position non-linear capacity | Like the MLP layers in your Module 003 networks |
| Positional encoding | Inject sequence order information | Replaces the inherent ordering in RNNs |

**Scale:** BERT-base stacks 12 of these blocks. GPT-3 stacks 96. Each additional block lets the model capture more abstract relationships.

> **Discussion:** What is the computational trade-off? Self-attention is $O(n^2)$ in sequence length (every token attends to every other). LSTMs are $O(n)$. For very long sequences (thousands of tokens), this quadratic cost becomes the bottleneck -- motivating architectures like Longformer and Mamba that approximate attention.

---
# STAGE 2 -- Pre-trained Models and Fine-tuning

**Connecting to Friday:** On Friday you fine-tuned MobileNet V2 (pre-trained on ImageNet) for CIFAR-10 image classification. You froze the convolutional feature extractor and retrained only the top classification head. Text transfer learning follows the identical pattern: take a transformer pre-trained on billions of words, freeze most layers, and adapt the top for your task.

**What you will learn:**
- How BERT, GPT, and T5 differ in architecture and training objective
- How to use Hugging Face `pipeline` for zero-effort inference
- When to fine-tune versus use off-the-shelf

## BERT vs GPT vs T5 -- Three Families of Pre-trained Transformers

| Feature | BERT | GPT | T5 |
|---------|------|-----|----|
| **Architecture** | Encoder only | Decoder only | Encoder-Decoder |
| **Attention direction** | Bidirectional (sees full context) | Causal / left-to-right only | Bidirectional encoder, causal decoder |
| **Training objective** | Masked Language Modeling (predict hidden words) | Causal Language Modeling (predict next word) | Text-to-Text (every task framed as text in, text out) |
| **Key insight** | Understanding context from both sides | Generating fluent text | Unifying all NLP tasks into one format |
| **Best for** | Classification, NER, question answering | Text generation, conversation, code | Summarization, translation, general-purpose |
| **Example sizes** | BERT-base: 110M params | GPT-2: 1.5B params | T5-base: 220M params |

**Connection to Friday's encoder-decoder:**
- BERT = just the encoder (like Friday's encoder, but with attention instead of LSTM)
- GPT = just the decoder (generates tokens autoregressively, like Friday's decoder)
- T5 = full encoder-decoder (like Friday's seq2seq, but with transformers)

> **Discussion:** Why would you choose an encoder-only model (BERT) over an encoder-decoder model (T5) for a classification task? (BERT is simpler, faster, and optimized for understanding. T5 can do classification but adds unnecessary decoder overhead. Choose the simplest architecture that solves your problem.)

## Transfer Learning for Text

The transfer learning pattern is identical across modalities:

| Step | Image (Friday) | Text (Today) |
|------|----------------|-------------|
| 1. Choose pre-trained model | MobileNet V2 (trained on ImageNet) | BERT-base (trained on BookCorpus + Wikipedia) |
| 2. What it already knows | Edges, textures, shapes, objects | Grammar, semantics, world knowledge |
| 3. Freeze base layers | Freeze convolutional feature extractor | Freeze transformer encoder layers |
| 4. Replace task head | New dense layer for 10 CIFAR classes | New dense layer for your classification task |
| 5. Fine-tune on your data | 50K CIFAR-10 images | Your task-specific labeled text |
| 6. Result | Image classifier for your domain | Text classifier for your domain |

**Why this works:** A model trained on billions of words has learned the structure of language itself -- grammar, common sense, factual knowledge. This foundation transfers to almost any text task, even with a small fine-tuning dataset.

**For FraudShield:** A BERT model pre-trained on general English can be fine-tuned on a few thousand labeled fraud reports to classify new reports as fraudulent or legitimate. The model already understands language; it just needs to learn what fraud language looks like.

## Hugging Face Pipeline: Sentiment Classification

The Hugging Face `transformers` library provides a `pipeline` API that handles tokenization, model loading, and inference in a single call. No training required -- these models are already fine-tuned.

> **What is happening in the next cell:** We load a pre-trained sentiment analysis model (DistilBERT fine-tuned on SST-2) and classify several example sentences. The model returns a label (POSITIVE/NEGATIVE) and a confidence score.

In [ ]:
from transformers import pipeline

sentiment_classifier = pipeline("sentiment-analysis")

test_sentences = [
    "The fraud detection system caught every suspicious transaction.",
    "Our model's recall dropped significantly after the data pipeline change.",
    "The new feature engineering approach improved F1 by 12 percent.",
    "Multiple legitimate transactions were incorrectly flagged as fraud.",
]

print("Sentiment Analysis Results")
print("=" * 70)
for sentence in test_sentences:
    result = sentiment_classifier(sentence)[0]
    print(f"  Label: {result['label']:>8}  |  Score: {result['score']:.4f}  |  {sentence}")

print()
print("Under the hood: text -> tokenizer -> DistilBERT -> [CLS] output -> classification head")
print("This model was fine-tuned on SST-2 (Stanford Sentiment Treebank).")
print("We provided zero training data -- we benefit from someone else's fine-tuning.")

## Hugging Face Pipeline: Text Generation

GPT-style models generate text autoregressively: they predict one token at a time, append it to the input, and predict again. This is the same autoregressive decoding from Friday's encoder-decoder, but without the encoder -- the model conditions only on the tokens generated so far.

> **What is happening in the next cell:** We load GPT-2 and generate a continuation of a prompt about fraud detection. The `max_new_tokens` parameter controls how many tokens the model generates. `temperature` controls randomness: lower values produce more deterministic output.

> **Discussion:** Could you use a text generation model for fraud detection directly? Why or why not?

In [ ]:
text_generator = pipeline("text-generation", model="gpt2")

prompt = "Machine learning models for fraud detection must balance"

result = text_generator(
    prompt,
    max_new_tokens=60,
    temperature=0.7,
    do_sample=True,
    num_return_sequences=1,
    pad_token_id=50256,
)

print("Text Generation (GPT-2)")
print("=" * 70)
print(f"Prompt: {prompt}")
print(f"\nGenerated:")
print(result[0]["generated_text"])
print()
print("Note: GPT-2 generates plausible continuations because it was trained on")
print("billions of words. The output is not factually guaranteed -- it is a")
print("statistical prediction of likely next tokens.")

## Hugging Face Pipeline: Summarization

Summarization uses an encoder-decoder transformer (T5 or BART). The encoder reads the full input text; the decoder generates a condensed version. This is exactly Friday's encoder-decoder architecture, but with attention replacing LSTM recurrence.

Two types of summarization:
- **Extractive:** select and copy important sentences from the original (simpler, but less flexible)
- **Abstractive:** generate new sentences that capture the meaning (what transformer models do)

> **What is happening in the next cell:** We summarize a paragraph about FraudShield's model evaluation process. The model reads the full text through its encoder and generates a shorter version through its decoder.

In [ ]:
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

long_text = (
    "The FraudShield risk analytics team deployed a Random Forest classifier "
    "trained on synthetic transaction data to detect fraudulent activity. The model "
    "was registered in the SageMaker Model Registry with metadata including the "
    "training job identifier, accuracy metrics, and the S3 path to the model artifact. "
    "After manual approval, the model was deployed to a real-time inference endpoint "
    "running on an ml.m5.xlarge instance. Evaluation on 400 held-out validation records "
    "showed strong precision but lower recall, indicating that while the model rarely "
    "generates false alarms, it misses a meaningful fraction of actual fraud cases. "
    "The team identified feature engineering improvements -- particularly incorporating "
    "transaction velocity and merchant category risk scores from the Feature Store -- "
    "as the most promising path to improving recall without sacrificing precision."
)

summary = summarizer(long_text, max_length=60, min_length=20, do_sample=False)

print("Summarization (DistilBART)")
print("=" * 70)
print(f"Original ({len(long_text.split())} words):")
print(f"  {long_text}")
print(f"\nSummary ({len(summary[0]['summary_text'].split())} words):")
print(f"  {summary[0]['summary_text']}")
print()
print("This is abstractive summarization -- the model generates new sentences")
print("rather than copying from the input.")

## When to Fine-tune vs Use Off-the-shelf

We have now used three off-the-shelf models without any training. When should you fine-tune instead?

| Scenario | Recommendation | Reasoning |
|----------|---------------|----------|
| Standard sentiment analysis | Off-the-shelf | Pre-trained models already handle this well |
| FraudShield transaction classification | Fine-tune | Domain-specific vocabulary and fraud patterns |
| Generating marketing copy | Off-the-shelf (large GPT) | General language generation is sufficient |
| Classifying support tickets into 47 categories | Fine-tune | Custom label set not in pre-training |
| Quick prototype or proof of concept | Off-the-shelf | Speed matters more than peak accuracy |
| Production system with strict accuracy targets | Fine-tune | Must optimize for specific data distribution |

**The decision framework:**

1. Does your task differ significantly from the pre-training distribution?
2. Do you have domain-specific vocabulary or patterns?
3. Do you have labeled data for your specific task?
4. Are accuracy requirements strict enough to justify the engineering cost?

If you answer "yes" to most of these, fine-tune. Otherwise, start with off-the-shelf and invest your engineering time elsewhere.

> **Discussion:** For FraudShield's document verification system (detecting forged IDs), would you fine-tune a text model or an image model? (Image model -- the task is visual, not textual. You would fine-tune a vision transformer or CNN, not BERT.)

---
# STAGE 3 -- Experiments, Lineage, and Reproducibility

**Connecting to prior days:**

- **Monday:** You deployed models and computed metrics manually. But which hyperparameters produced which metrics? You would need to scroll through CloudWatch logs to find out.
- **Tuesday:** You created Feature Store feature groups. But which models were trained on which features? Without tracking, that connection is lost.

SageMaker Experiments solves the tracking problem. It organizes training runs into experiments, logs every parameter and metric, and connects to lineage tracking so you can trace any model back to the exact data, features, and code that produced it.

**What you will learn:**
- How to create experiments, runs, and log metrics/parameters/artifacts
- How to compare multiple runs to find the best configuration
- How lineage tracking connects data, processing, training, and deployment
- How Feature Store integrates with lineage
- Reproducibility patterns for ML workflows

## SageMaker Experiments Concepts

Think of SageMaker Experiments as a structured lab notebook.

| Concept | Analogy | Example |
|---------|---------|--------|
| **Experiment** | A research project | "FraudShield Transaction Classifier" |
| **Run** | One attempt in that project | "RF with 100 trees, feature set A" |
| **Parameters** | What you configured | `n_estimators=100`, `max_depth=10` |
| **Metrics** | What you measured | `accuracy=0.92`, `f1=0.85` |
| **Artifacts** | What you produced | `model.tar.gz`, `confusion_matrix.png` |

Every training job in SageMaker can automatically log to an experiment run. You can also log manually for local experiments or custom workflows.

**Why this matters:** Without structured tracking, you end up with dozens of models in S3, CloudWatch logs scattered across training jobs, and no way to answer "which configuration produced the best F1 score?" Experiments answer that question with a single API call.

> **What is happening in the next cells:** We create an experiment, create two runs with different configurations, log parameters and metrics, and then compare the runs.

In [ ]:
from sagemaker.experiments.run import Run
from datetime import datetime

sm_client = boto3.client("sagemaker")

TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
EXPERIMENT_NAME = f"fraudshield-transformer-study-{TIMESTAMP}"

with Run(
    experiment_name=EXPERIMENT_NAME,
    run_name="rf-100-trees-baseline",
    sagemaker_session=session,
) as run:
    run.log_parameter("model_type", "RandomForest")
    run.log_parameter("n_estimators", 100)
    run.log_parameter("max_depth", "None")
    run.log_parameter("random_state", 42)
    run.log_parameter("feature_count", 6)
    run.log_parameter("feature_source", "synthetic_v1")

    run.log_metric("accuracy", 0.9200)
    run.log_metric("precision", 0.9500)
    run.log_metric("recall", 0.6000)
    run.log_metric("f1", 0.7350)

    print(f"Experiment: {EXPERIMENT_NAME}")
    print(f"Run:        rf-100-trees-baseline")
    print(f"")
    print(f"Logged parameters:")
    print(f"  model_type:     RandomForest")
    print(f"  n_estimators:   100")
    print(f"  max_depth:      None")
    print(f"  random_state:   42")
    print(f"  feature_count:  6")
    print(f"  feature_source: synthetic_v1")
    print(f"")
    print(f"Logged metrics:")
    print(f"  accuracy:  0.9200")
    print(f"  precision: 0.9500")
    print(f"  recall:    0.6000")
    print(f"  f1:        0.7350")

In [ ]:
with Run(
    experiment_name=EXPERIMENT_NAME,
    run_name="rf-200-trees-tuned",
    sagemaker_session=session,
) as run:
    run.log_parameter("model_type", "RandomForest")
    run.log_parameter("n_estimators", 200)
    run.log_parameter("max_depth", 10)
    run.log_parameter("random_state", 42)
    run.log_parameter("feature_count", 8)
    run.log_parameter("feature_source", "feature_store_v1")

    run.log_metric("accuracy", 0.9425)
    run.log_metric("precision", 0.9300)
    run.log_metric("recall", 0.7200)
    run.log_metric("f1", 0.8117)

    print(f"Run:        rf-200-trees-tuned")
    print(f"")
    print(f"Logged parameters:")
    print(f"  model_type:     RandomForest")
    print(f"  n_estimators:   200")
    print(f"  max_depth:      10")
    print(f"  random_state:   42")
    print(f"  feature_count:  8")
    print(f"  feature_source: feature_store_v1")
    print(f"")
    print(f"Logged metrics:")
    print(f"  accuracy:  0.9425")
    print(f"  precision: 0.9300")
    print(f"  recall:    0.7200")
    print(f"  f1:        0.8117")

## Compare Experiment Runs

The value of experiments becomes clear when you have multiple runs. Instead of scrolling through CloudWatch logs or maintaining a spreadsheet, you query the Experiments API to list, filter, sort, and compare runs programmatically.

SageMaker Studio also provides a visual comparison UI where you can select runs and view their parameters and metrics side by side.

> **What is happening in the next cell:** We use the `ExperimentAnalytics` API to list all runs in our experiment, extract their parameters and metrics, and display a comparison table. We then identify the best run by F1 score.

> **Discussion:** Besides hyperparameters, what else should you vary across experiment runs? (Feature sets from Feature Store, preprocessing steps, training data versions, model architectures, sampling strategies.)

In [ ]:
from sagemaker.analytics import ExperimentAnalytics
import pandas as pd

experiment_analytics = ExperimentAnalytics(
    experiment_name=EXPERIMENT_NAME,
    sagemaker_session=session,
)

runs_df = experiment_analytics.dataframe()

if runs_df.empty:
    print("No runs found yet. ExperimentAnalytics may take a moment to index.")
    print("If this persists, the runs were logged successfully -- the analytics")
    print("index is eventually consistent.")
else:
    display_cols = [col for col in runs_df.columns
                    if any(kw in col.lower() for kw in
                           ["run", "accuracy", "precision", "recall", "f1",
                            "n_estimators", "max_depth", "feature"])]
    if display_cols:
        print("Experiment Runs Comparison")
        print("=" * 70)
        print(runs_df[display_cols].to_string(index=False))
    else:
        print("Experiment Runs (all columns):")
        print(runs_df.to_string(index=False))

print(f"\nTotal runs in experiment: {len(runs_df)}")
print(f"\nIn production with 50+ runs, you would sort by F1 to find the best")
print(f"configuration, then inspect its parameters to understand what worked.")

## Lineage Tracking Concepts

Experiments track **what you tried** and **what happened**. Lineage tracking goes further: it records the **causal chain** from raw data to deployed endpoint.

```
Data Source (S3 bucket)
    |
    v
Processing Job (Data Wrangler / Processing)
    |
    v
Feature Group (Feature Store)          <-- Tuesday's work
    |
    v
Training Dataset
    |
    v
Training Job (Experiment Run)          <-- Today's Stage 3
    |
    v
Model Artifact (model.tar.gz in S3)
    |
    v
Model Package (Model Registry)         <-- Monday's work
    |
    v
Endpoint (Deployed Model)              <-- Monday's work
```

**Why lineage matters:** If a deployed model starts producing bad predictions, lineage lets you trace back through every step: which training job produced it, what data was used, what features were selected, what preprocessing was applied. Without lineage, debugging a production model is detective work.

**SageMaker lineage entities:**

| Entity | Role | Example |
|--------|------|--------|
| **Context** | Groups related entities | An experiment, a pipeline execution |
| **Action** | A process that transforms data | A training job, a processing job |
| **Artifact** | A data object | A dataset in S3, a model artifact, an endpoint |
| **Association** | A link between entities | "Training job X produced model Y" |

SageMaker automatically creates lineage entities for training jobs, processing jobs, and endpoints. You can also create custom lineage entities for steps that happen outside SageMaker.

## Feature Store Lineage Integration

**Connecting to Tuesday:** On Tuesday you created feature groups in SageMaker Feature Store -- `transaction_features`, `merchant_features`, etc. Those feature groups are lineage entities. When a training job reads from a feature group, SageMaker records that relationship.

The connection looks like this:

```
Feature Group: "transaction_features"     (created Tuesday)
        |
        | [ContributedTo]
        v
Training Job: "rf-100-trees-baseline"     (logged today)
        |
        | [Produced]
        v
Model Artifact: model.tar.gz
```

This enables powerful queries:

- **Forward tracing:** "Which models were trained on `transaction_features`?" -- Find all downstream consumers of a feature group.
- **Backward tracing:** "What data produced this deployed model?" -- Trace from endpoint back to feature groups and raw data.
- **Impact analysis:** "If I change the `amount` normalization in `transaction_features`, which models are affected?" -- Identify all downstream dependencies before making changes.

**Cross-account sharing** (from your CT reading) extends this lineage across organizational boundaries. A central feature team can share feature groups with multiple ML teams, and the lineage tracks which team's models consume which features.

> **Discussion:** Why is backward tracing critical for regulated industries (banking, healthcare)? (Regulators may require you to explain exactly how a model was built -- what data went in, how it was processed, what features were used, and what the model's accuracy was on validation data. Lineage provides this audit trail automatically.)

## Reproducibility Patterns

Lineage tells you **what happened**. Reproducibility ensures you can **make it happen again**.

| Pattern | What to Track | SageMaker Mechanism |
|---------|--------------|--------------------|
| **Data versioning** | Exact dataset used for training | S3 versioning + Feature Store point-in-time queries |
| **Code versioning** | Training script, preprocessing code | Git commit hash logged as experiment parameter |
| **Environment versioning** | Container image, library versions | Docker image URI logged with training job |
| **Hyperparameter logging** | All configuration values | Experiment Run parameters |
| **Random seed control** | All sources of randomness | Logged as hyperparameter |
| **Metric logging** | All evaluation results | Experiment Run metrics |
| **Artifact tracking** | Model files, plots, reports | Experiment Run artifacts + S3 |

**Minimum reproducibility checklist:**

1. Pin your random seeds (NumPy, Python, framework-specific)
2. Version your data (S3 versioning or Feature Store point-in-time queries)
3. Log your code commit (Git hash as an experiment parameter)
4. Record your container image (Docker URI from the training job)
5. Log all hyperparameters (every configurable value)

If you do these five things, any colleague can recreate your exact training run six months later.

> **Discussion:** What happens if you reproduce a training run but get slightly different results? What could cause that? (Non-deterministic GPU operations like cuDNN autotuning, floating-point ordering differences in parallel reductions, library version mismatches, different hardware. Some randomness is inherent in distributed GPU training.)

## Cleanup: Delete Experiment Runs

Just as we delete endpoints after use, we clean up experiment runs when they are no longer needed. In production you would keep runs indefinitely for audit and comparison. For this training exercise, we clean up to avoid clutter.

> **What is happening in the next cell:** We list all runs in our experiment, delete each run, then delete the experiment itself.

In [ ]:
import time

print(f"Cleaning up experiment: {EXPERIMENT_NAME}")
print()

try:
    trials = sm_client.list_trial_components(
        ExperimentName=EXPERIMENT_NAME
    ).get("TrialComponentSummaries", [])

    for tc in trials:
        tc_name = tc["TrialComponentName"]
        associations = sm_client.list_associations(
            SourceArn=tc["TrialComponentArn"]
        ).get("AssociationSummaries", [])
        for assoc in associations:
            sm_client.delete_association(
                SourceArn=assoc["SourceArn"],
                DestinationArn=assoc["DestinationArn"],
            )

        trials_for_tc = sm_client.list_trials(
            ExperimentName=EXPERIMENT_NAME
        ).get("TrialSummaries", [])
        for trial in trials_for_tc:
            try:
                sm_client.disassociate_trial_component(
                    TrialComponentName=tc_name,
                    TrialName=trial["TrialName"],
                )
            except Exception:
                pass

        sm_client.delete_trial_component(TrialComponentName=tc_name)
        print(f"  Deleted trial component: {tc_name}")

    trials_list = sm_client.list_trials(
        ExperimentName=EXPERIMENT_NAME
    ).get("TrialSummaries", [])
    for trial in trials_list:
        sm_client.delete_trial(TrialName=trial["TrialName"])
        print(f"  Deleted trial: {trial['TrialName']}")

    sm_client.delete_experiment(ExperimentName=EXPERIMENT_NAME)
    print(f"\nDeleted experiment: {EXPERIMENT_NAME}")

except Exception as e:
    print(f"Cleanup note: {e}")
    print("If the experiment was already deleted or does not exist, this is expected.")

print("\nCleanup complete.")

---
## Wrap-up and Thursday Preview

**What you accomplished today:**

1. **Built attention from scratch:** Scaled dot-product attention in NumPy, extended to multi-head attention, added positional encoding, and assembled a complete transformer encoder block. You understand why self-attention replaces sequential LSTM recurrence with parallel computation.

2. **Used pre-trained transformers:** BERT for classification, GPT-2 for generation, DistilBART for summarization -- all through Hugging Face pipelines. The transfer learning pattern is identical to Friday's MobileNet: large model pre-trained on generic data, adapted to specific tasks.

3. **Tracked experiments and lineage:** Created SageMaker Experiments, logged parameters and metrics, compared runs, and connected lineage to Tuesday's Feature Store. You can now trace any model back to the exact data, features, and code that produced it.

**Thursday preview:**

Thursday covers **NLP pipelines and model optimization**. You will tokenize text, build end-to-end NLP workflows, and learn techniques for making models faster and smaller for production deployment -- quantization, pruning, and distillation. The question shifts from "what architecture should I use" to "how do I make it production-ready."

Read the **NLP Pipelines** and **Model Optimization** concept threads before Thursday.